# modular-diffusion: the clean engine with the prompt, in the mixed assembly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.16.2/colab/sd15_clean_with_prompt.ipynb)

The pairs notebook (`colab/three_band_pairs.ipynb`) drew a MIXED assembly of Stable Diffusion 1.5 band engines: the first capture's clean engine and the middle engine of the shape `2222/nomid/3`, the noisy engine of `tiny`. Its pictures are as close to real ones as the model's own, but they match their prompts less well (the CLIP text-to-picture score about 1.2 under the model's). Its clean engine was trained WITHOUT the prompt. This notebook trains that engine again WITH the prompt, every other setting the same, and draws the mixed assembly with it. A training step at batch 16 peaks at 39 to 57 GiB in one forward. The second cell times it on this card in one forward and, if that does not fit, in groups of 8, 4 or 2 rows (each group's forward and backward alone, the gradients added, one optimizer step: the same update in less memory); the trainings run with the first that fits.

1. THE TRAINING: the clean engine of `2222/nomid/3`, inherited from the model, 3,000 steps at batch 16, read every 250 steps on 64 held-out rows, with the prompt read.
2. A PARITY CHECK: the stored mixed assembly drawn again on this card on the first 100 held-out prompts, against its stored pictures.
3. THE NEW ENGINE ALONE (the model in the two other stretches) on the first 500 held-out prompts, against the model's own pictures, the real ones and the stored pictures of the engine without the prompt.
4. THE MIXED ASSEMBLY WITH IT on all 1,000 held-out prompts, against the model's pictures, the real ones and the stored mixed assembly (row by row: the same prompts and start noise).
5. The new engine's distance read on its stretch.
6. LAST, A TWIN WITHOUT THE PROMPT ON THE SAME ROWS: the same training with the prompt not read (the 4,000 training pictures, the same seed, this card), drawn alone and in the mixed assembly like the new engine, with its distance read. The stored engine without the prompt was trained on another set of rows, so the twin separates the two differences: the new engine against the twin is the prompt at equal rows (row by row: the same prompts and start noise), the twin against the stored engine is the rows.

Every picture job: start noise from seed 2000 in groups of 50, 50 DDIM steps, guidance 7.5, the parts in half precision; Inception features (the kernel and Frechet distances) and the CLIP score; a sheet of its first 16 pictures; and a `weights.json` that names each part's folder, whether it reads the prompt and the hashes of its weight files.

The cells: (1) install at a release tag; (2) the card checked, the stored jobs brought back from the Hub (the cell stops if one is missing), the model, the probe, judge and training latents, the CLIP judge, a smoke run of every job on a tiny random model, the real judges on 16 pictures, 30 training steps of the new form and of the old one timed with their peak memory (in one forward first, then in smaller groups of rows until a try fits under 85% of the card; the cell stops if none does), one picture job and one distance read through the new engine's path, and the projected minutes; (3) the job list: each job runs and its folder is shipped at once, the report is shipped, and the machine is given back only when everything finished was shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`; on a new machine they come back from the Hub with their files, so nothing runs twice.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[capture,hub,data,guard,judge] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.16.2"
import os, sys, time, json, gc, hashlib, shutil, statistics, threading
import numpy as np
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 16, 2), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import diffusers, transformers, huggingface_hub
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/sd15_clean_with_prompt"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The card first: its name and memory are printed. Then the stored jobs this notebook reads come back from the Hub by name (the three engines of the mixed assembly, the real pictures' and the model's pictures, the stored mixed assembly's pictures, the stored pictures of the clean engine without the prompt); if one does not, the cell stops before anything else. As the three-band notebook: the model at a pinned revision, both validation parquet files of COCO 2017, the PROBE (the first 256 pictures, their content hash checked), the JUDGE latents (the first 1,000 pictures) and the TRAINING latents (the 4,000 other pictures), each written once and skipped when present; the CLIP model at a pinned revision. The engines brought back are loaded once and checked (their shape, whether they read the prompt). The smoke runs every job of the list on a tiny random model, with small stand-ins of the stored jobs, and checks that the new engine's files say it reads the prompt, that the stored assembly drawn again gives its own pictures, and that every picture job names the trained engine and its weights' hashes. The real judges run on 16 pictures with the judge's own control; the smoke, the license copy and the notice are shipped. Then 30 training steps of the new form and of the old one are timed on this card with their peak memory: in one forward first (`MICRO` 0), then in groups of 8, 4 and 2 rows, a try that runs out of memory or peaks over 85% of the card moving to the next; the first that fits is `MICRO`, the two trainings of the job list run with it, and the cell stops if none fits (`MICRO_FORCE` at the top of the cell forces a value), and one picture job of 16 rows and one distance read of 16 rows run through the new engine's path as the roster runs a job, which measures a job's fixed start-up apart from its work; the projection adds them up.

In [ ]:
# C2 - the card, the stored jobs, the model, the data, the checks, the size of the job list
MICRO_FORCE = None      # the training's group size: None = chosen by the timed smoke below; a number forces it (0 = the whole batch)
from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files
import subprocess
import pyarrow.parquet as pq
from geolip.modular_diffusion import hub, run
from geolip.modular_diffusion.capture import BANDS, pictures
from geolip.modular_diffusion.capture.assemble import Assembly
from geolip.modular_diffusion.capture.data import (LatentBatches, center_crop, encode_shards, parquet_pictures, read_index,
                                                   rows_in_order, synthetic_shards)
from geolip.modular_diffusion.capture.engine import load_engine
from geolip.modular_diffusion.capture.teacher import load_teacher
from geolip.modular_diffusion.capture.train import train_band_engine
from geolip.modular_diffusion.experiments import capture
TEACHER, PROBE, JUDGE, TRAIN, RUNS = f"{OUT}/sd15", f"{OUT}/probe", f"{OUT}/judge", f"{OUT}/train", f"{OUT}/runs"
FIRST, CLIP = f"{OUT}/first_capture", f"{OUT}/clip"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/capture/three_band"
FIRST_PREFIX = "modular_diffusion/capture/first_capture"
NOT_SHIPPED = ("checkpoint.pt", "SHIPPED", "FROM_HUB")           # the resume checkpoint and two markers; the rest ships
os.makedirs(RUNS, exist_ok=True)

# the card: its name and memory. A training step of the larger shape at batch 16 peaks at 39 to 57 GiB in one forward;
# in groups of MICRO rows (each group's forward and backward alone, the gradients added, one optimizer step: the same
# update) it needs less. MICRO is chosen further down by timing the training on this card
MAX_PEAK_SHARE = 0.85      # a try whose peak memory is over this share of the card moves to a smaller group
assert torch.cuda.is_available(), "no card: choose a GPU runtime"
CARD = torch.cuda.get_device_properties(0)
CARD_GIB = CARD.total_memory / 2**30
print(f"card: {CARD.name} | memory {CARD_GIB:.1f} GiB | the training's group size is chosen by the timed smoke below")

# the stored jobs this measurement reads, from the results repository: the three engines of the mixed assembly (the
# first capture's clean engine without the prompt, the middle engine of 2222/nomid/3 and the noisy engine of tiny), the
# real pictures' and the model's own pictures, the stored mixed assembly's pictures and the stored pictures of the
# clean engine without the prompt beside the model; each comes back with its files, else the cell stops here
STORED_ENGINE_FILES = ("engine.json", "engine.safetensors", "engine_avg.safetensors", "train.json", "result.json")
STORED_PICTURE_FILES = ("pictures.npz", "result.json")
FIRST_CLEAN, MIDDLE, NOISY = "clean_2222nomid3_inherit_s0", "middle_2222nomid3_inherit_s0", "noisy_tiny_inherit_s0"
KEPT = {MIDDLE: STORED_ENGINE_FILES, NOISY: STORED_ENGINE_FILES, "pic_rows_1000": STORED_PICTURE_FILES,
        "pic_teacher_1000": STORED_PICTURE_FILES, "pic_mixed_s0_1000": STORED_PICTURE_FILES,
        "pic_clean_2222nomid3_s0_500": STORED_PICTURE_FILES}


def fetch_stored(local=f"{OUT}/hub_stored"):
    """Every stored job's files into its folder (the first capture's engine under FIRST, the rest under RUNS), marked
    DONE and FROM_HUB; a job already marked is not fetched again. Returns the jobs that did not come back."""
    wanted = [(FIRST_PREFIX, FIRST_CLEAN, STORED_ENGINE_FILES, f"{FIRST}/{FIRST_CLEAN}")]
    wanted += [(PREFIX, t, files, f"{RUNS}/{t}") for t, files in KEPT.items()]
    missing = []
    for prefix, job, files, dst in wanted:
        if os.path.exists(f"{dst}/FROM_HUB"):
            continue
        try:
            for name in files:
                got = hf_hub_download(repo_id=REPO, filename=f"{prefix}/{job}/{name}", local_dir=local)
                os.makedirs(dst, exist_ok=True)
                shutil.copyfile(got, f"{dst}/{name}")
        except Exception as e:
            missing.append(f"{job} ({repr(e)[:160]})")
            continue
        for m in ("DONE", "FROM_HUB"):
            with open(f"{dst}/{m}", "w", encoding="utf-8") as fh:
                fh.write("fetched back from the Hub\n")
    return missing


MISSING = fetch_stored()
assert not MISSING, "a stored job this measurement reads did not come back from the Hub: " + "; ".join(MISSING)
print(f"the stored jobs: {1 + len(KEPT)} came back from the Hub ({FIRST_CLEAN} from the first capture; " + ", ".join(KEPT) + ")")

# the data
snapshot_download(repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5", revision="451f4fe16113bff5a5d2269ed5ad43b0592e9a14",
                  allow_patterns=["model_index.json", "scheduler/scheduler_config.json", "tokenizer/*", "text_encoder/config.json",
                                  "text_encoder/model.safetensors", "unet/config.json", "unet/diffusion_pytorch_model.safetensors",
                                  "vae/config.json", "vae/diffusion_pytorch_model.safetensors"], local_dir=TEACHER)
PARQUETS = [hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset", revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff",
                            filename=f"data/val-0000{i}-of-00002.parquet", local_dir=f"{OUT}/coco") for i in (0, 1)]
N_PROBE, N_JUDGE = 256, 1000

def content_hash(pictures, captions, size=512):
    h = hashlib.sha256()
    for p, c in zip(pictures, captions):
        a = ((center_crop(p, size) + 1.0) * 127.5).round().clamp(0, 255).to(torch.uint8)    # the array the encoder sees
        h.update(a.numpy().tobytes())
        h.update(c.encode("utf-8") + b"\0")
    return h.hexdigest()

def first_captions(path):
    out = []        # the first caption of every row, read without the pictures (the rule of parquet_pictures)
    for b in pq.ParquetFile(path).iter_batches(batch_size=512, columns=["answer"]):
        for a in b.column(0).to_pylist():
            out.append("" if a is None else str((a[0] if a else "") if isinstance(a, (list, tuple)) else a))
    return out

EXPECTED_CONTENT_HASH = "93d333a65f7bbdac6ced9143ff55852273c4e2a64deb7f8f641aeb1878f35c44"           # the 256 probe pictures of the band-needs run and the first capture

def slices(sources):
    # 512 pictures at a time (decoded, handed to the encoder one by one, then dropped), so memory stays flat
    for path, first_row, end in sources:
        total = pq.ParquetFile(path).metadata.num_rows if end is None else end
        for first in range(first_row, total, 512):
            pics, _, _ = parquet_pictures(path, min(512, total - first), skip=first)
            yield from pics
            del pics

JUDGE_SOURCES = [(PARQUETS[0], 0, N_JUDGE)]
TRAIN_SOURCES = [(PARQUETS[0], N_JUDGE, None), (PARQUETS[1], 0, None)]
captions_of = lambda sources: [c for path, first, end in sources for c in first_captions(path)[first:end]]
judge_captions, train_captions = captions_of(JUDGE_SOURCES), captions_of(TRAIN_SOURCES)
JUDGE_ROWS, TRAIN_ROWS = len(judge_captions), len(train_captions)
print("judge pictures:", JUDGE_ROWS, "| expected 1,000; training pictures:", TRAIN_ROWS, "| expected 4,000")
assert JUDGE_ROWS == N_JUDGE and TRAIN_ROWS == 4000, "the judge or training rows are not the expected ones"
ready = lambda folder, n: os.path.exists(f"{folder}/index.json") and sum(read_index(folder)["counts"]) == n
probe_ready, judge_ready, train_ready = ready(PROBE, N_PROBE), ready(JUDGE, JUDGE_ROWS), ready(TRAIN, TRAIN_ROWS)
if not (probe_ready and judge_ready and train_ready):
    with_vae = load_teacher(TEACHER, device="cuda", dtype=torch.float32, vae=True)
    if not probe_ready:
        pictures_, captions, meta = parquet_pictures(PARQUETS[0], N_PROBE)
        HASH = content_hash(pictures_, captions)
        print("probe content hash", HASH, "| expected", EXPECTED_CONTENT_HASH)
        assert HASH == EXPECTED_CONTENT_HASH, "the probe pictures are not the expected data"
        encode_shards(pictures_, captions, PROBE, with_vae.vae, with_vae.tokenize, shard_size=64, batch=16, size=512, device="cuda")
        json.dump({"content_hash": HASH, "pictures": [{"row": i, "id": m.get("id"), "license": m.get("license")} for i, m in enumerate(meta)]},
                  open(f"{PROBE}/pictures.json", "w", encoding="utf-8"), indent=1)
        del pictures_
    if not judge_ready:
        # encode_shards is given a generator of pictures; a stopped encoding continues at its first unfinished shard
        encode_shards(slices(JUDGE_SOURCES), judge_captions, JUDGE, with_vae.vae, with_vae.tokenize, shard_size=500, batch=16,
                      size=512, device="cuda")
    if not train_ready:
        encode_shards(slices(TRAIN_SOURCES), train_captions, TRAIN, with_vae.vae, with_vae.tokenize, shard_size=512, batch=16,
                      size=512, device="cuda")
    del with_vae
    gc.collect()
    torch.cuda.empty_cache()
assert sum(read_index(JUDGE)["counts"]) == N_JUDGE and sum(read_index(TRAIN)["counts"]) == TRAIN_ROWS
print("probe shards:", read_index(PROBE)["counts"], "| judge shards:", read_index(JUDGE)["counts"], "| training shards:",
      read_index(TRAIN)["counts"])

# the engines brought back, loaded once on the CPU: each is the shape it was trained as, and reads the prompt as it did
cpu_teacher = load_teacher(TEACHER, device="cpu", dtype=torch.float32)
for folder, want_shape, want_text in ((f"{FIRST}/{FIRST_CLEAN}", "2222/nomid/3", False), (f"{RUNS}/{MIDDLE}", "2222/nomid/3", True),
                                      (f"{RUNS}/{NOISY}", "tiny", True)):
    e = load_engine(cpu_teacher, folder, average=True)
    print(f"{os.path.basename(folder)}: shape {e.shape.name}, reads the prompt: {e.text}")
    assert e.shape.name == want_shape and e.text is want_text, f"{folder} is not the stored engine it should be"
    del e
del cpu_teacher
gc.collect()

# the job list: (tag, arguments, the jobs it waits for, kind), in the order of what each job gates; the stored jobs it
# reads are under `runs` and the first capture's clean engine under `first` (inputs, not jobs of this list)
COMMON = (f"teacher={TEACHER} device=cuda threads=4 probe_data={PROBE} probe_first=0 probe_rows=64 probe_seed=1000 "
          "probe_per_band=4 chunk=16")
DISTANCE = "dtype=fp16 sample_steps=50 guidance=7.5 group=16"     # the teacher in half precision; the engines stay float32
PICTURES = (f"variant=pictures dtype=fp16 part_dtype=fp16 pic_data={JUDGE} pic_first=0 sample_steps=50 guidance=7.5 "
            f"judge=inception clip={CLIP} sheet_n=16")
TEACHER_PARTS = "clean=teacher,middle=teacher,noisy=teacher"
NEW = "clean_2222nomid3_inherit_text_s0"          # the clean engine of 2222/nomid/3 trained again, reading the prompt
# the first capture's recipe for its clean engine of this shape, argument for argument, except text=1 (the prompt read)
NEW_TRAINING = ("variant=band_engine band=clean shape=2222/nomid/3 init=inherit text=1 steps=3000 lr=5e-05 batch=16 drop=0.1 "
                f"seed=0 autocast=bf16 read_every=250 keep_checkpoint=0 dtype=fp32 data={TRAIN}")
TWIN = "clean_2222nomid3_inherit_notext_s0"       # its twin without the prompt: the same job, the same rows, text=0
assert NEW_TRAINING.count(" text=1 ") == 1
TWIN_TRAINING = NEW_TRAINING.replace(" text=1 ", " text=0 ")
# the recipe the stored middle and noisy engines were trained by (the smoke makes small stand-ins of them)
STORED_TRAINING = f"variant=band_engine init=inherit text=1 batch=16 lr=5e-5 autocast=bf16 read_every=250 keep_checkpoint=0 data={TRAIN} dtype=fp32"
WEIGHT_KINDS = ("train", "pictures", "sample")     # the jobs whose folders get weights.json before they ship


def mixed_parts(runs, clean):
    return f"clean={clean},middle={runs}/{MIDDLE},noisy={runs}/{NOISY}"


def stored_jobs(runs, first, rows=(1000, 500), group=50):
    """The stored jobs this measurement reads, as they were run (they come back from the Hub; the smoke runs them
    small); the first capture's clean engine is not among them (the smoke trains its stand-in apart)."""
    big, half = rows
    pic = f"{PICTURES} group={group}"
    return [(MIDDLE, f"{STORED_TRAINING} shape=2222/nomid/3 band=middle steps=3000 seed=0", [], "train"),
            (NOISY, f"{STORED_TRAINING} shape=tiny band=noisy steps=3000 seed=0", [], "train"),
            ("pic_rows_1000", f"{pic} source=rows pic_n={big}", [], "pictures"),
            ("pic_teacher_1000", f"{pic} parts={TEACHER_PARTS} pic_n={big} noise_seed=2000", [], "pictures"),
            ("pic_mixed_s0_1000", f"{pic} parts={mixed_parts(runs, f'{first}/{FIRST_CLEAN}')} pic_n={big} noise_seed=2000",
             [MIDDLE, NOISY], "pictures"),
            ("pic_clean_2222nomid3_s0_500", f"{pic} parts=clean={first}/{FIRST_CLEAN},middle=teacher,noisy=teacher "
             f"pic_n={half} noise_seed=2000", [], "pictures")]


def job_list(runs, first, rows=(1000, 500, 100), blocks=10, group=50):
    at = lambda t: f"{runs}/{t}"
    big, half, small = rows
    pic = f"{PICTURES} group={group}"
    cmp = f"variant=picture_compare blocks={blocks}"
    jobs = []

    def compare(name, a, b):
        jobs.append((name, f"{cmp} a={at(a)} b={at(b)}", [a, b], "compare"))

    # (1) the training: everything below reads its engine
    jobs.append((NEW, NEW_TRAINING, [], "train"))
    # (2) the parity of this card: the stored mixed assembly drawn again on the first rows, against its stored pictures
    jobs.append(("pic_mixed_s0_100_again", f"{pic} parts={mixed_parts(runs, f'{first}/{FIRST_CLEAN}')} pic_n={small} "
                 "noise_seed=2000", [MIDDLE, NOISY], "pictures"))
    compare("cmp_parity_mixed_100", "pic_mixed_s0_100_again", "pic_mixed_s0_1000")
    # (3) the new clean engine alone (the model in the other two stretches), against the model, the real pictures, and
    # the stored pictures of the clean engine without the prompt
    jobs.append(("pic_clean_text_2222nomid3_s0_500", f"{pic} parts=clean={at(NEW)},middle=teacher,noisy=teacher pic_n={half} "
                 "noise_seed=2000", [NEW], "pictures"))
    compare("cmp_clean_text_2222nomid3_s0_500_vs_teacher", "pic_clean_text_2222nomid3_s0_500", "pic_teacher_1000")
    compare("cmp_clean_text_2222nomid3_s0_500_vs_rows", "pic_clean_text_2222nomid3_s0_500", "pic_rows_1000")
    compare("cmp_clean_text_vs_clean_2222nomid3_s0_500", "pic_clean_text_2222nomid3_s0_500", "pic_clean_2222nomid3_s0_500")
    # (4) the mixed assembly with it, against the model, the real pictures and the stored mixed assembly (row by row)
    jobs.append(("pic_mixed_text_s0_1000", f"{pic} parts={mixed_parts(runs, at(NEW))} pic_n={big} noise_seed=2000",
                 [NEW, MIDDLE, NOISY], "pictures"))
    compare("cmp_mixed_text_s0_1000_vs_teacher", "pic_mixed_text_s0_1000", "pic_teacher_1000")
    compare("cmp_mixed_text_s0_1000_vs_rows", "pic_mixed_text_s0_1000", "pic_rows_1000")
    compare("cmp_mixed_text_vs_mixed_s0_1000", "pic_mixed_text_s0_1000", "pic_mixed_s0_1000")
    # (5) the new engine's distance read on its stretch (the model in the other two), as the first capture's
    jobs.append(("asm_clean_text_2222nomid3_s0", f"variant=assembly parts=clean={at(NEW)},middle=teacher,noisy=teacher {DISTANCE}",
                 [NEW], "sample"))
    # (6) LAST, so that the reads above are in first: a twin WITHOUT the prompt on the same rows (the new job's arguments
    # with text=0); the new engine minus the twin is the prompt at equal rows, the twin minus the stored engine the rows
    jobs.append((TWIN, TWIN_TRAINING, [NEW], "train"))    # waits for the new training: if it fails, the twin is not run
    jobs.append(("pic_clean_notext_2222nomid3_s0_500", f"{pic} parts=clean={at(TWIN)},middle=teacher,noisy=teacher pic_n={half} "
                 "noise_seed=2000", [TWIN], "pictures"))
    compare("cmp_clean_notext_2222nomid3_s0_500_vs_teacher", "pic_clean_notext_2222nomid3_s0_500", "pic_teacher_1000")
    compare("cmp_clean_notext_2222nomid3_s0_500_vs_rows", "pic_clean_notext_2222nomid3_s0_500", "pic_rows_1000")
    compare("cmp_clean_text_vs_notext_2222nomid3_s0_500", "pic_clean_text_2222nomid3_s0_500", "pic_clean_notext_2222nomid3_s0_500")
    compare("cmp_clean_notext_vs_clean_2222nomid3_s0_500", "pic_clean_notext_2222nomid3_s0_500", "pic_clean_2222nomid3_s0_500")
    jobs.append(("pic_mixed_notext_s0_1000", f"{pic} parts={mixed_parts(runs, at(TWIN))} pic_n={big} noise_seed=2000",
                 [TWIN, MIDDLE, NOISY], "pictures"))
    compare("cmp_mixed_notext_s0_1000_vs_teacher", "pic_mixed_notext_s0_1000", "pic_teacher_1000")
    compare("cmp_mixed_notext_s0_1000_vs_rows", "pic_mixed_notext_s0_1000", "pic_rows_1000")
    compare("cmp_mixed_text_vs_notext_s0_1000", "pic_mixed_text_s0_1000", "pic_mixed_notext_s0_1000")
    compare("cmp_mixed_notext_vs_mixed_s0_1000", "pic_mixed_notext_s0_1000", "pic_mixed_s0_1000")
    jobs.append(("asm_clean_notext_2222nomid3_s0", f"variant=assembly parts=clean={at(TWIN)},middle=teacher,noisy=teacher {DISTANCE}",
                 [TWIN], "sample"))
    stored = [t for t, *_ in stored_jobs(runs, first)]
    for i, (t, args, needs, kind) in enumerate(jobs):    # every folder a job reads is a job before it, or a stored one
        assert all(n in stored or n in [j[0] for j in jobs[:i]] for n in needs), t
    return jobs


# what a job was given: each part's folder, whether it reads the prompt and the content hashes of its two weight files,
# written into the job's folder as weights.json before it ships (the picture results name a part's folder only)
HASHES = {}


def file_hash(path):
    st = os.stat(path)
    key = (path, st.st_size, st.st_mtime_ns)
    if key not in HASHES:
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 24), b""):
                h.update(chunk)
        HASHES[key] = h.hexdigest()
    return HASHES[key]


def part_record(where):
    if where == "teacher" or where.startswith("born:"):
        return {"part": where}
    card = json.load(open(f"{where}/engine.json", encoding="utf-8"))
    rec = {"part": where, "job": os.path.basename(where.rstrip("/")), "text": bool(card["text"]), "shape": card.get("shape_name")}
    for name in ("engine.safetensors", "engine_avg.safetensors"):
        path = f"{where}/{name}"
        rec[name.replace(".safetensors", "_sha256")] = file_hash(path) if os.path.exists(path) else None
    return rec


def write_weights(folder):
    r = json.load(open(f"{folder}/result.json", encoding="utf-8"))
    parts = r.get("parts") or r.get("args", {}).get("parts", "")
    if r.get("variant") == "band_engine":
        rec = {"engine": part_record(folder)}
    elif parts:
        spec = dict(x.split("=", 1) for x in parts.split(","))
        rec = {"parts": {b: part_record(spec[b]) for b in BANDS}}
    else:
        return None
    with open(f"{folder}/weights.json", "w", encoding="utf-8", newline="\n") as fh:
        json.dump(rec, fh, indent=1)
    return rec

# the text-to-picture judge at a pinned revision: its settings and weights only
snapshot_download(repo_id="openai/clip-vit-large-patch14", revision="32bd64288804d66eefd0ccbe215aa642df71cc41",
                  allow_patterns=["config.json", "model.safetensors", "preprocessor_config.json"], local_dir=CLIP)

# the smoke: every job of the list on the tiny random model, a few steps each, with stand-ins of the stored jobs it
# reads (the first capture's clean engine by a two-step tiny engine without the prompt) and the checks that apply
DEVICE = "cuda"
SMOKE = f"{OUT}/smoke_clean_text"
shutil.rmtree(SMOKE, ignore_errors=True)
synthetic_shards(f"{SMOKE}/rows", rows=8, shard_size=3, size=16, seed=4)       # the tiny model's 16 x 16 latents
synthetic_shards(f"{SMOKE}/judge", rows=20, shard_size=7, size=16, seed=5)
SMOKE_ROWS, SMOKE_BLOCKS, SMOKE_GROUP = (20, 10, 4), 2, 2      # at least 2 rows a block of the kernel distance
SMALL = (f" tiny=1 device={DEVICE} probe_data={SMOKE}/rows probe_rows=4 probe_per_band=2 chunk=2 steps=4 batch=2 read_every=2"
         f" sample_steps=4 group={SMOKE_GROUP} save_every=0 log_every=0 pic_data={SMOKE}/judge judge=fake clip=fake sheet_n=4"
         + ("" if DEVICE == "cuda" else " autocast=none"))
run.main("capture", (f"variant=band_engine band=clean text=0 init=inherit shape=2222/nomid/3 seed=0 {COMMON}{SMALL} steps=2").split()
         + [f"out={SMOKE}/first_capture/{FIRST_CLEAN}"])
smoke_jobs = (stored_jobs(f"{SMOKE}/runs", f"{SMOKE}/first_capture", SMOKE_ROWS[:2], SMOKE_GROUP)
              + job_list(f"{SMOKE}/runs", f"{SMOKE}/first_capture", SMOKE_ROWS, SMOKE_BLOCKS, SMOKE_GROUP))
S, W = {}, {}
for t, args, needs, kind in smoke_jobs:
    S[t] = run.main("capture", f"{args} {COMMON}{SMALL}".split() + [f"out={SMOKE}/runs/{t}"])
    if kind in WEIGHT_KINDS:
        W[t] = write_weights(f"{SMOKE}/runs/{t}")
card_new = json.load(open(f"{SMOKE}/runs/{NEW}/engine.json", encoding="utf-8"))
settings_new = json.load(open(f"{SMOKE}/runs/{NEW}/train.json", encoding="utf-8"))["settings"]
assert card_new["text"] is True and settings_new["text"] is True and S[NEW]["reads"]["born"]["text"] is True, \
    "the new training's files do not say that its engine reads the prompt"
assert len(S[NEW]["curve"]) == 3 and len(S[MIDDLE]["curve"]) == 3, "a training has no read curve"
assert S["cmp_parity_mixed_100"]["features_equal"] is True, "the stored assembly drawn again on this card is not its pictures"
new_rec = W[NEW]["engine"]
for t in ("pic_clean_text_2222nomid3_s0_500", "pic_mixed_text_s0_1000", "asm_clean_text_2222nomid3_s0"):
    got = W[t]["parts"]["clean"]
    assert got["job"] == NEW and got["text"] is True and got["engine_avg_sha256"] == new_rec["engine_avg_sha256"] \
        and got["engine_sha256"] == new_rec["engine_sha256"], f"{t} was not given the trained engine"
assert W["pic_mixed_s0_100_again"]["parts"]["clean"]["text"] is False, "the parity job's clean part reads the prompt"
card_twin = json.load(open(f"{SMOKE}/runs/{TWIN}/engine.json", encoding="utf-8"))
assert card_twin["text"] is False and S[TWIN]["reads"]["born"]["text"] is False, "the twin's files do not say it ignores the prompt"
twin_rec = W[TWIN]["engine"]
for t in ("pic_clean_notext_2222nomid3_s0_500", "pic_mixed_notext_s0_1000", "asm_clean_notext_2222nomid3_s0"):
    got = W[t]["parts"]["clean"]
    assert got["job"] == TWIN and got["text"] is False and got["engine_avg_sha256"] == twin_rec["engine_avg_sha256"] \
        and got["engine_sha256"] == twin_rec["engine_sha256"], f"{t} was not given the twin"
print(f"smoke: {len(smoke_jobs) + 1} jobs on the tiny model; the new engine reads the prompt by its files; the stored "
      "assembly drawn again gives its pictures; each picture job names the trained engine and its weights' hashes")

# the real judges on the real model, 16 rows of the judge latents; then the judge's own control: the same pictures
# scored against the captions shifted by one row must score lower than against their own
JUDGE_KIND, CLIP_KIND, STEPS16, REAL_EXTRA = "inception", CLIP, 50, ""
FEATURE_DIM = 2048 if JUDGE_KIND == "inception" else 64
J16_OUT = f"{OUT}/judges16"
shutil.rmtree(J16_OUT, ignore_errors=True)
J16 = run.main("capture", (f"variant=pictures teacher={TEACHER} device={DEVICE} threads=4 parts={TEACHER_PARTS} pic_data={JUDGE} "
                           f"pic_first=0 pic_n=16 group=16 noise_seed=2000 sample_steps={STEPS16} guidance=7.5 judge={JUDGE_KIND} "
                           f"clip={CLIP_KIND} dtype=fp16 part_dtype=fp16 sheet_n=16{REAL_EXTRA}").split() + [f"out={J16_OUT}"])
with np.load(f"{J16_OUT}/pictures.npz") as d:
    feats16, clip16 = d["features"], d["clip"]
assert feats16.shape == (16, FEATURE_DIM) and np.isfinite(feats16).all(), f"the judge's features: {feats16.shape}"
assert np.isfinite(clip16).all(), "a text-to-picture score is not finite"
t16 = load_teacher(TEACHER, device=DEVICE, dtype=torch.float16, vae=True)
t16.vae.float()
x16, ids16 = rows_in_order(JUDGE, 0, 16)
shape16 = (int(t16.unet.config.in_channels), int(t16.unet.config.sample_size), int(t16.unet.config.sample_size))
z16 = pictures.sample_latents(Assembly({b: t16 for b in BANDS}), t16.text_states(ids16), t16.empty_states(1), 2000, group=16,
                              steps=STEPS16, guidance=7.5, latent_shape=shape16, alphas_cumprod=t16.alphas_cumprod)
pics16 = pictures.decode(t16.vae, z16)
clip_judge = pictures.make_clip_judge(CLIP_KIND, DEVICE)
own, shifted = clip_judge.scores(pics16, ids16), clip_judge.scores(pics16, ids16.roll(1, 0))
print(f"real judges, 16 pictures of the model: features {tuple(feats16.shape)}; CLIP against their own captions "
      f"{own.mean():.3f} (the job: {clip16.mean():.3f}), against the captions shifted by one row {shifted.mean():.3f}")
assert abs(float(own.mean()) - float(clip16.mean())) < 0.05, "the cell's pictures are not the job's"
assert shifted.mean() < own.mean(), "the CLIP judge does not score pictures higher against their own captions"
ROW_S = J16["seconds_sampling"] / 16                      # seconds a row of a sampled picture job
print(f"sampling: {ROW_S:.2f} s a row (16 rows, 50 steps, guidance 7.5, half precision)")
del t16, z16, pics16, clip_judge
gc.collect()
torch.cuda.empty_cache()

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    # with the tiny engines' weights, so the upload of large files is checked too
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke_clean_text", "the clean engine with the prompt: smoke run", ignore=NOT_SHIPPED), \
        "the smoke folder was not shipped: check the token and the repository before the long run"

# the license copy and a notice, shipped to the root of the results folder before any trained weights
LIC = hf_hub_download(repo_id="CompVis/stable-diffusion-license", repo_type="space", filename="license.txt", local_dir=f"{OUT}/license")
NOTICE = f"{OUT}/notice"
os.makedirs(NOTICE, exist_ok=True)
shutil.copyfile(LIC, f"{NOTICE}/LICENSE_STABLE_DIFFUSION_1_5.txt")       # byte for byte
lic_bytes = open(f"{NOTICE}/LICENSE_STABLE_DIFFUSION_1_5.txt", "rb").read()
print("license copy: sha256", hashlib.sha256(lic_bytes).hexdigest(), "|", len(lic_bytes), "bytes")
NOTICE_TEXT = (
    "# Notice: files made from Stable Diffusion 1.5\n\n"
    "The `engine.safetensors` and `engine_avg.safetensors` files in the folders beside this notice are band engines: reduced "
    "copies of the Stable Diffusion 1.5 U-Net (`stable-diffusion-v1-5/stable-diffusion-v1-5`, revision `451f4fe1`). Some start "
    "from that model's weights, and all were trained on its answers or in its latent space, so they are derivatives of that "
    "model.\n\n"
    "Stable Diffusion 1.5 is published under the CreativeML Open RAIL-M license. A copy of that license is in this folder "
    "(`LICENSE_STABLE_DIFFUSION_1_5.txt`); its use restrictions (its Attachment A) travel with these files as they do with the "
    "model.\n\n"
    "These are research artifacts of a measurement, not a finished model: each engine covers one stretch of the noise range and "
    "is read beside the original model (one engine of this folder, the one whose name starts with `whole_`, was trained on "
    "the whole range for comparison).\n")
with open(f"{NOTICE}/NOTICE.md", "w", encoding="utf-8", newline="\n") as fh:
    fh.write(NOTICE_TEXT)
AT_PREFIX = set(list_repo_files(REPO)) if REPO else set()      # shipped by an earlier run; shipped again only if missing
if REPO and not {f"{PREFIX}/LICENSE_STABLE_DIFFUSION_1_5.txt", f"{PREFIX}/NOTICE.md"} <= AT_PREFIX:
    assert hub.ship(NOTICE, REPO, PREFIX, "the clean engine with the prompt: license copy and notice"), "the license copy and notice were not shipped"
else:
    print("the license copy and the notice are already in the results folder: not shipped again")

# the training on this card: TIMING_STEPS steps of the new job's form (the clean stretch, 2222/nomid/3, batch 16, the
# prompt read) and as many of the stored engine's form (the same without the prompt), each step stamped (the card
# synchronized); the seconds a step = the median gap of steps 11 onward; the peak memory of each beside the card's.
# THE GROUP SIZE: both forms are tried with the whole batch in one forward (MICRO 0), then in groups of 8, 4 and 2 rows;
# a try that runs out of memory, or whose peak is over MAX_PEAK_SHARE of the card, moves to the next; the first that
# fits is MICRO (MICRO_FORCE set: that value only), and the job list's trainings run with it
TIMING_STEPS = 30
AUTOCAST = "bf16" if DEVICE == "cuda" else "none"
teacher = load_teacher(TEACHER, device=DEVICE, dtype=torch.float32)
source = LatentBatches(TRAIN, seed=0)


def free_card():
    gc.collect()                        # what the try before left is freed first, so this try's peak is its own
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def timed(text, micro):
    """TIMING_STEPS training steps of one form in groups of `micro` rows -> (seconds a step, peak memory or None,
    trained numbers)."""
    folder = f"{OUT}/timing/text{text}"
    shutil.rmtree(folder, ignore_errors=True)
    stamps = []

    def note(line):
        if line.startswith("step "):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            stamps.append(time.time())
    free_card()
    job = train_band_engine(teacher, "2222/nomid/3", "clean", source, steps=TIMING_STEPS, batch=16, seed=0, out=folder,
                            text=bool(text), autocast=AUTOCAST, save_every=0, log_every=1, log=note, keep_checkpoint=False,
                            micro=micro)
    sec = statistics.median([b - a for a, b in zip(stamps[9:-1], stamps[10:])])
    peak = None
    if torch.cuda.is_available():
        peak = {"reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
                "allocated_gib": torch.cuda.max_memory_allocated() / 2**30}
    return sec, peak, int(job["trained"])


MICRO, TRIES = None, []
for micro in ((MICRO_FORCE,) if MICRO_FORCE is not None else (0, 8, 4, 2)):
    SEC, PEAK, TRAINED, why = {}, {}, {}, None
    try:
        for text in (1, 0):
            sec, peak, n = timed(text, micro)
            SEC[f"text{text}"], TRAINED[f"text{text}"] = sec, n
            if peak is not None:
                PEAK[f"text{text}"] = peak
                if peak["reserved_gib"] > MAX_PEAK_SHARE * CARD_GIB:
                    why = (f"the prompt {'read' if text else 'not read'}: peak {peak['reserved_gib']:.1f} GiB, over "
                           f"{MAX_PEAK_SHARE:.0%} of the card's {CARD_GIB:.1f} GiB")
                    break
    except torch.cuda.OutOfMemoryError as e:
        why = "out of memory: " + str(e).split("\n")[0][:160]
    TRIES.append({"micro": micro, "fits": why is None, "why": why})
    if why is None:
        MICRO = micro
        break
    print(f"micro {micro} (groups of {micro or 16} rows): does not fit ({why}); the next group size")
    free_card()
assert MICRO is not None, (f"no group size fits this card ({CARD_GIB:.1f} GiB; tried " + ", ".join(
    f"micro {t['micro']}: {t['why']}" for t in TRIES) + "): choose a larger card")
for text in (1, 0):
    k = f"text{text}"
    if k in PEAK:
        print(f"training, the prompt {'read' if text else 'not read'}: {SEC[k]:.3f} s a step at batch 16 in groups of "
              f"{MICRO or 16} rows (the median gap of steps 11 to {TIMING_STEPS}: the first ten steps left out); peak "
              f"memory {PEAK[k]['reserved_gib']:.1f} GiB reserved ({PEAK[k]['allocated_gib']:.1f} GiB allocated) of the "
              f"card's {CARD_GIB:.1f} GiB")
    else:
        print(f"training, the prompt {'read' if text else 'not read'}: {SEC[k]:.3f} s a step at batch 16 in groups of "
              f"{MICRO or 16} rows (the median gap of steps 11 to {TIMING_STEPS}: the first ten steps left out; no card)")
    print(f"  trained numbers: {TRAINED[k]:,}")
if MICRO > 0:                    # the job list's two trainings run in groups of MICRO rows (the same update)
    NEW_TRAINING = f"{NEW_TRAINING} micro={MICRO}"
    TWIN_TRAINING = NEW_TRAINING.replace(" text=1 ", " text=0 ")
shutil.rmtree(f"{OUT}/timing/text0", ignore_errors=True)
del teacher, source
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
SMOKE_ENGINE = f"{OUT}/timing/text1"        # the 30-step engine of the new form: the two smokes below read it

# a job's fixed start-up apart from its work: 16 rows of the mixed assembly with that engine in the clean stretch, run
# as the roster runs a job (a new process); its start-up = its wall time minus its sampling, decoding and judging
JOB_ENV = dict(os.environ, OMP_NUM_THREADS="4", MKL_NUM_THREADS="4", HF_TOKEN="", PYTHONUTF8="1", PYTHONUNBUFFERED="1")


def as_a_job(args, out):
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out)
    t = time.time()
    with open(f"{out}/run.log", "wb") as fh:
        p = subprocess.run([sys.executable, "-u", "-m", "geolip.modular_diffusion.run", "capture"] + args.split() + [f"out={out}"],
                           env=JOB_ENV, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)
    wall = time.time() - t
    tail = open(f"{out}/run.log", encoding="utf-8", errors="replace").read()[-2000:]
    assert p.returncode == 0, f"the smoke job in {out} failed:\n{tail}"
    return json.load(open(f"{out}/result.json", encoding="utf-8")), wall


PIC_SMOKE_ROWS, DIST_SMOKE_ROWS = 16, 16
PIC_SMOKE, PIC_WALL = as_a_job(f"{PICTURES} group=16 parts={mixed_parts(RUNS, SMOKE_ENGINE)} pic_n={PIC_SMOKE_ROWS} noise_seed=2000 "
                               f"{COMMON} device={DEVICE} sample_steps={STEPS16} judge={JUDGE_KIND} clip={CLIP_KIND}{REAL_EXTRA}",
                               f"{OUT}/pic_smoke")
PIC_WORK = PIC_SMOKE["seconds_sampling"] + PIC_SMOKE["seconds_decoding"] + PIC_SMOKE["seconds_judging"]
STARTUP_S = max(0.0, PIC_WALL - PIC_WORK)
pic_rec = write_weights(f"{OUT}/pic_smoke")["parts"]
assert pic_rec["clean"]["part"] == SMOKE_ENGINE and pic_rec["clean"]["text"] is True, "the smoke picture job did not read the new engine"
assert pic_rec["middle"]["job"] == MIDDLE and pic_rec["noisy"]["job"] == NOISY, "the smoke picture job did not read the stored engines"
# the distance read's path: the same engine on its stretch, the model in the other two, on DIST_SMOKE_ROWS probe rows
DIST_SMOKE, DIST_WALL = as_a_job(f"variant=assembly parts=clean={SMOKE_ENGINE},middle=teacher,noisy=teacher {DISTANCE} {COMMON} "
                                 f"device={DEVICE} probe_rows={DIST_SMOKE_ROWS} sample_steps={STEPS16}{REAL_EXTRA}", f"{OUT}/dist_smoke")
assert all(DIST_SMOKE["reads"][b]["mse"] >= 0 for b in BANDS) and DIST_SMOKE["final_latent_change"] >= 0
ROW_WORK = max((J16["seconds_sampling"] + J16["seconds_decoding"] + J16["seconds_judging"]) / 16,   # the model's own rows
               PIC_WORK / PIC_SMOKE_ROWS)                                                          # the mixed assembly's
DIST_S = STARTUP_S + max(0.0, DIST_WALL - STARTUP_S) * 64 / DIST_SMOKE_ROWS
print(f"a job's fixed start-up on this card: {STARTUP_S:.0f} s (a new process, the imports, the model and the parts loaded); "
      f"a picture row {ROW_WORK:.2f} s of work (sampling, decoding, judging: the larger of the model's 16 rows and the mixed "
      f"assembly's {PIC_WORK / PIC_SMOKE_ROWS:.2f} s); a distance read of the 64 probe rows about {DIST_S / 60:.1f} min")
shutil.rmtree(f"{OUT}/timing", ignore_errors=True)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# each job's cost from those numbers: the start-up once a job; a training adds its steps at the measured seconds a step
# and its reads (29 band reads and the model's answers on three stretches, 16 forwards of 16 rows each, a forward taken
# as a quarter of a training step); a picture job adds its rows at the measured work a row (the real pictures' rows a
# tenth of a second each: decoding and judging only); a distance read the measured one scaled to 64 rows
def cost(args, kind):
    a = dict(x.split("=", 1) for x in args.split() if "=" in x)
    if kind == "compare":
        return 15.0
    if kind == "train":
        steps, sec = int(a["steps"]), SEC["text0" if a["text"] == "0" else "text1"]
        reads = (2 * (steps // int(a["read_every"])) + 5 + 3) * 16 * sec / 4
        return STARTUP_S + steps * sec + reads
    if kind == "sample":
        return DIST_S
    n = int(a["pic_n"])
    return STARTUP_S + n * (0.1 if a.get("source") == "rows" else ROW_WORK)


JOBS = job_list(RUNS, FIRST)
COST = {t: cost(a, k) for t, a, _, k in JOBS}
assert len({t for t, *_ in JOBS}) == len(JOBS) and all(n in COST or n in KEPT for _, _, ns, _ in JOBS for n in ns)
COUNTS = {k: sum(kk == k for *_, kk in JOBS) for k in ("train", "sample", "pictures", "compare")}
assert (COUNTS["train"], COUNTS["sample"], COUNTS["pictures"], COUNTS["compare"]) == (2, 2, 5, 15), COUNTS
TWIN_FIRST = [t for t, *_ in JOBS].index(TWIN)
TWIN_MIN = sum(COST[t] for t, *_ in JOBS[TWIN_FIRST:]) / 60      # the twin's jobs, the last of the list
START_PART = STARTUP_S * sum(k != "compare" for *_, k in JOBS)
INSTRUMENTS = {"card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "card_gib": CARD_GIB,
               "seconds_a_step_batch16": SEC, "timing_steps": TIMING_STEPS, "micro": MICRO, "micro_tries": TRIES, "peak_memory": PEAK,
               # the price of the prompt on ONE card: the same steps with and without it (shape, band, batch the same)
               "step_seconds_text1": SEC["text1"], "step_seconds_text0": SEC["text0"],
               "peak_gib_text1": PEAK.get("text1", {}).get("reserved_gib"), "peak_gib_text0": PEAK.get("text0", {}).get("reserved_gib"),
               "trained_text1": TRAINED["text1"], "trained_text0": TRAINED["text0"],
               "peak_share_allowed": MAX_PEAK_SHARE, "job_startup_seconds": STARTUP_S, "picture_row_work_seconds": ROW_WORK,
               "mixed_picture_row_work_seconds": PIC_WORK / PIC_SMOKE_ROWS, "distance_read_seconds_64_rows": DIST_S,
               "smoke_picture_weights": pic_rec, "training_rows": TRAIN_ROWS, "judge_rows": JUDGE_ROWS,
               "real_judges_16": {"features": list(feats16.shape), "clip_own": float(own.mean()),
                                  "clip_shifted_by_one_row": float(shifted.mean())}}
TRAIN_STEPS_MIN = sum(int(dict(x.split("=", 1) for x in a.split() if "=" in x)["steps"])
                      * SEC["text0" if " text=0 " in f" {a} " else "text1"] for t, a, _, k in JOBS if k == "train") / 60
print(f"the job list: {len(JOBS)} jobs ({COUNTS['train']} trainings, {COUNTS['pictures']} picture jobs, {COUNTS['sample']} "
      f"distance reads, {COUNTS['compare']} comparisons), about {sum(COST.values()) / 60:.0f} projected minutes on this card with "
      f"one runner: the start-ups {START_PART / 60:.0f}, the trainings' steps {TRAIN_STEPS_MIN:.0f}, the picture rows "
      f"{sum(COST[t] - STARTUP_S for t, *_, k in JOBS if k == 'pictures') / 60:.0f}, the distance reads "
      f"{COUNTS['sample'] * (DIST_S - STARTUP_S) / 60:.0f}, the comparisons {15 * COUNTS['compare'] / 60:.0f}; of these the twin "
      f"without the prompt (run last) about {TWIN_MIN:.0f}")

print(f"card {INSTRUMENTS['card']} | {CARD_GIB:.1f} GiB | MICRO {MICRO} | the smoke's peak "
      + (f"{max(p['reserved_gib'] for p in PEAK.values()):.1f} GiB" if PEAK else "not read (no card)")
      + f" | {SEC['text1']:.3f} s a step with the prompt, {SEC['text0']:.3f} without | about {sum(COST.values()) / 60:.0f} "
      "projected minutes")

One runner works the job list in order: the training first, then the parity check, the new engine alone and its comparisons, the mixed assembly with it and its comparisons, the distance read, and last the twin without the prompt with its pictures, its comparisons (against the model, the real pictures, the new engine and the stored engine) and its distance read. Each finished job folder gets its `weights.json` and is shipped at once to `REPO` beside the earlier runs' jobs, with the token from the notebook's secret `HF_TOKEN`. At the end the summary, the roster, its log and the instrument numbers are shipped to `_report_clean_text`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the job list, each finished job shipped, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
EXTRA = ""                         # more arguments for every job (empty in a real run)
PROGRESS_S = 60
kind = {t: k for t, _, _, k in JOBS}
needs = {t: n for t, _, n, _ in JOBS}
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | capture {a} {COMMON}{EXTRA}" + (f" | needs={','.join(n)}" if n else "")
                       for t, a, n, _ in JOBS) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
ENGINE_FILES = ("engine.safetensors", "engine_avg.safetensors", "engine.json", "train.json")
PICTURE_FILES = ("pictures.npz",)

def fetch_back():
    # on a new machine: finished jobs come back with their files, so nothing runs twice; a job comes back only when what
    # it reads came back too
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if kind[t] == "train" and not all(os.path.exists(f"{src}/{f}") for f in ENGINE_FILES):
            continue                                             # its engine files are not on the Hub: it is trained again
        if kind[t] == "pictures" and not all(os.path.exists(f"{src}/{f}") for f in PICTURE_FILES):
            continue                                             # its features are not on the Hub: it is run again
        if not all(mark(n, "DONE") for n in needs[t]):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

if REPO:
    fetch_back()
assert all(mark(t, "DONE") for t in KEPT), ("a stored job this measurement reads is not on this machine (run the second "
                                            "cell again): " + ", ".join(t for t in KEPT if not mark(t, "DONE")))
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and kind[tag] in WEIGHT_KINDS:
        write_weights(folder)      # each part's folder, whether it reads the prompt, its weights' hashes
    if ok and REPO:
        if hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"the clean engine with the prompt: {tag}", ignore=NOT_SHIPPED):
            open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")      # kept across sessions on this machine
        else:
            print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
runner.start()
while runner.is_alive():
    runner.join(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = capture.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{os.path.basename(f).replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report_clean_text", "the clean engine with the prompt: summary, roster and checks") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none")
if RELEASE and shipped and not unshipped:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else "RELEASE is False")